# CheckThat Task 3 — Stage 4 Defensive Article Generator

**Version:** `DEFENSIVE_FILTER_MANUAL_CITATIONS_v2`

This notebook reads:

```text
outputs/stage3/stage3_packets_{DATASET_TAG}.json
```

and writes:

```text
outputs/stage4/stage4_articles_{DATASET_TAG}.json
outputs/submission/prediction_{DATASET_TAG}.json
outputs/submission/prediction.json
```

Design goals:

- Treat `must_cite_evidence` and `optional_context` as **candidate evidence**, not mandatory citations.
- Defensively filter Stage 3 packets before generation.
- Avoid weak context, duplicate claim-origin sources, no-content pages, boilerplate, and sources promoted only to satisfy minimum count.
- Preserve verdict nuance for `mostly_true`, `half_true`, and `misleading`.
- Lead false claims with strongest refutation/provenance/contradiction.
- **Prevent citation hallucination by making the model output evidence IDs only; Python appends exact URLs.**
- Prefer one URL per sentence because the official citation precision script penalizes unnecessary multi-source citations.
- Support resume, parallel generation, repair, validation, and inspection.


## 1. Install, authenticate, and create Vertex Gemini client

In [ ]:
!pip install -q -U google-genai

from google.colab import auth, drive
auth.authenticate_user()
drive.mount("/content/drive")

import os
from google import genai
from google.genai import types

PROJECT_ID = "clef-checkthat"
LOCATION = "global"

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
)

print("Vertex Gemini client ready.")

## 2. Configuration

In [ ]:
from pathlib import Path

DRIVE_BASE = Path("/content/drive/MyDrive/CheckThat_Task3_Dataset")

# Examples: "TARGET10", "100_test", "train", "dev"
DATASET_TAG = "TARGET10"

STAGE3_IN = DRIVE_BASE / f"outputs/stage3/stage3_packets_{DATASET_TAG}.json"

LOCAL_STAGE4_DIR = Path("/content/outputs/stage4")
LOCAL_STAGE4_DIR.mkdir(parents=True, exist_ok=True)
STAGE4_OUT = LOCAL_STAGE4_DIR / f"stage4_articles_{DATASET_TAG}.json"

DRIVE_STAGE4_DIR = DRIVE_BASE / "outputs/stage4"
DRIVE_STAGE4_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_STAGE4_OUT = DRIVE_STAGE4_DIR / STAGE4_OUT.name

SUBMISSION_DIR = DRIVE_BASE / "outputs/submission"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
PREDICTION_TAGGED_OUT = SUBMISSION_DIR / f"prediction_{DATASET_TAG}.json"
PREDICTION_CANONICAL_OUT = SUBMISSION_DIR / "prediction.json"

# Stage 4 model. If unavailable in your Vertex project, try "gemini-3-flash-preview",
# "gemini-2.5-pro", or "gemini-2.5-flash".
STAGE4_MODEL = "gemini-3-pro-preview"
STAGE4_FALLBACK_MODELS = ["gemini-3-flash-preview", "gemini-2.5-pro", "gemini-2.5-flash"]

STAGE4_TEMPERATURE = 0.15
STAGE4_REPAIR_TEMPERATURE = 0.0
STAGE4_MAX_OUTPUT_TOKENS = 4096
STAGE4_THINKING_LEVEL = "low"  # "low", "medium", "high", or None

MAX_RETRIES = 3
MAX_WORKERS = 6
SAVE_AFTER_EVERY_COMPLETED = 1

# Evidence selection settings.
MAX_STAGE4_EVIDENCE = 6
MIN_STRONG_SELECTED_FOR_REPAIR = 1

# Do not force a minimum evidence count. This is the point of defensive Stage 4.
ALLOW_LOW_EVIDENCE_ARTICLES = True

# Optional debugging controls.
TARGET_CLAIM_IDS = {str(i) for i in range(30349, 30359)}
ENFORCE_TARGET_CLAIM_IDS = True

# Set to an integer for a quick API cap, or None for the full selected set.
MAX_API_CALLS = None

# Existing outputs are reused unless their IDs are listed here.
FORCE_REGENERATE_IDS = set()

# A single ID to inspect in the later cells.
INSPECT_ID = "30356"

print("Dataset tag:", DATASET_TAG)
print("Stage 3 input:", STAGE3_IN)
print("Stage 4 local output:", STAGE4_OUT)
print("Stage 4 Drive output:", DRIVE_STAGE4_OUT)
print("Submission tagged output:", PREDICTION_TAGGED_OUT)
print("Submission canonical output:", PREDICTION_CANONICAL_OUT)
print("Model:", STAGE4_MODEL)
print("MAX_WORKERS:", MAX_WORKERS)

## 3. Utilities

In [ ]:
import json
import re
import time
import random
import shutil
from collections import Counter, defaultdict
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path


def load_json(path):
    path = Path(path)
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def save_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    tmp.replace(path)


def copy_to_drive(local_path, drive_path):
    drive_path = Path(drive_path)
    drive_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(local_path, drive_path)


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def clean_text(x, max_len=None):
    s = re.sub(r"\s+", " ", str(x or "")).strip()
    if max_len and len(s) > max_len:
        s = s[:max_len].rstrip() + "..."
    return s


def safe_int(x, default=0):
    try:
        return int(x)
    except Exception:
        return default


def listify(x):
    if x is None:
        return []
    if isinstance(x, list):
        return x
    return [x]


def unique_keep_order(items):
    out = []
    seen = set()
    for item in items:
        if item not in seen:
            seen.add(item)
            out.append(item)
    return out


def normalize_rating(value):
    raw = clean_text(value).lower().replace("-", "_").replace(" ", "_")
    raw = re.sub(r"_+", "_", raw)

    aliases = {
        "true": "true",
        "correct": "true",
        "mostly_true": "mostly_true",
        "partly_true": "half_true",
        "half_true": "half_true",
        "mixture": "half_true",
        "mixed": "half_true",
        "misleading": "misleading",
        "false": "false",
        "incorrect": "false",
        "pants_on_fire": "pants_fire",
        "pants_fire": "pants_fire",
        "fake": "pants_fire",
        "satire": "satire",
        "satirical": "satire",
    }
    return aliases.get(raw, raw or "unknown")


def claim_id(x):
    return str(x.get("id", "")).strip()


def get_response_text(response):
    text = getattr(response, "text", None)
    if text:
        return text.strip()

    chunks = []
    for cand in getattr(response, "candidates", None) or []:
        content = getattr(cand, "content", None)
        parts = getattr(content, "parts", None) if content else None
        for part in parts or []:
            part_text = getattr(part, "text", None)
            if part_text:
                chunks.append(part_text)
    return "\n".join(chunks).strip()


def extract_json_response(response):
    parsed = getattr(response, "parsed", None)
    if parsed:
        if isinstance(parsed, dict):
            return parsed
        try:
            return json.loads(json.dumps(parsed))
        except Exception:
            pass

    raw = get_response_text(response)
    if not raw:
        raise ValueError("Empty model response")

    raw = re.sub(r"^```(?:json)?\s*", "", raw.strip(), flags=re.IGNORECASE)
    raw = re.sub(r"\s*```$", "", raw).strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        start = raw.find("{")
        end = raw.rfind("}")
        if start >= 0 and end > start:
            return json.loads(raw[start:end + 1])
        raise


def response_usage_summary(response):
    usage = getattr(response, "usage_metadata", None)
    if usage is None:
        return {}
    return {
        "prompt_token_count": getattr(usage, "prompt_token_count", None),
        "candidates_token_count": getattr(usage, "candidates_token_count", None),
        "thoughts_token_count": getattr(usage, "thoughts_token_count", None),
        "total_token_count": getattr(usage, "total_token_count", None),
    }

## 4. Load Stage 3 packets

In [ ]:
assert STAGE3_IN.exists(), f"Missing Stage 3 input: {STAGE3_IN}"

stage3_packets = load_json(STAGE3_IN)

if ENFORCE_TARGET_CLAIM_IDS:
    stage3_packets = [p for p in stage3_packets if claim_id(p) in TARGET_CLAIM_IDS]

stage3_packets = sorted(stage3_packets, key=lambda p: claim_id(p))

print("Stage 3 packets loaded:", len(stage3_packets))
print("Claim IDs:", [claim_id(p) for p in stage3_packets])

if ENFORCE_TARGET_CLAIM_IDS:
    found = {claim_id(p) for p in stage3_packets}
    missing = sorted(TARGET_CLAIM_IDS - found)
    if missing:
        print("WARNING: Missing target IDs from Stage 3:", missing)

## 5. Defensive evidence filtering

In [ ]:
CORE_ROLES = {"core_support", "core_refutation"}
PROVENANCE_ROLES = {"visual_origin", "visual_misattribution", "image_manipulation"}
ORIGIN_ROLES = {"claim_origin", "satirical_origin", "fake_news_origin"}
CAVEAT_ROLES = {"temporal_refutation", "rating_caveat"}
CONTEXT_ROLES = {"legal_or_definition_context", "scientific_context", "background_context"}
STAT_ROLES = {"statistical_evidence"}
BAD_ROLES = {"off_topic", "no_content"}

NUANCE_RATINGS = {"mostly_true", "half_true", "misleading"}
FALSE_RATINGS = {"false", "pants_fire"}

HARD_EXCLUDE_FLAG_PREFIXES = {
    "off_topic_or_boilerplate",
    "no_readable_content",
}

SOFT_EXCLUDE_FLAG_PREFIXES = {
    "weak_context_only",
    "boilerplate_or_sidebar",
}

WARNING_FLAG_PREFIXES = {
    "claim_origin_not_proof",
    "visual_origin_only",
    "statistical_exactness_needed",
    "rating_caveat",
    "temporal_mismatch",
    "entity_mismatch",
    "location_mismatch",
    "date_mismatch",
    "predicate_mismatch",
    "number_unit_mismatch",
    "trial_vs_approval_mismatch",
    "visited_vs_played_mismatch",
    "satire_vs_factual_mismatch",
    "image_misattribution",
    "image_manipulation",
    "fake_news_origin",
}


def flag_prefix(flag):
    return clean_text(flag).split(":", 1)[0].strip().lower()


def item_text_blob(item):
    parts = [item.get("safe_source_fact", "")]
    for snip in item.get("snippets", []) or []:
        if isinstance(snip, dict):
            parts.append(snip.get("text", ""))
    return clean_text(" ".join(parts))


def item_has_usable_text(item):
    return bool(item_text_blob(item))


def source_usage_rule(item):
    role = clean_text(item.get("evidence_role"))
    relation = clean_text(item.get("source_relation"))
    stance = clean_text(item.get("stance_to_claim"))

    if role == "claim_origin":
        return "claim_origin_only: cite only to show that the claim was made or circulated; never use as proof."
    if role in {"satirical_origin", "fake_news_origin"}:
        return "origin_refutation: cite to explain satire/fake/parody provenance; never use as proof the real-world claim is true."
    if role in PROVENANCE_ROLES:
        return "visual_provenance: cite to explain visual origin, misattribution, or manipulation."
    if role in CAVEAT_ROLES:
        return "caveat: cite to qualify the claim or preserve rating nuance."
    if role in CONTEXT_ROLES:
        return "context_only: cite only if essential for understanding; do not use as direct proof."
    if role in STAT_ROLES:
        return "statistical: cite only exact numbers, units, and framing present in this evidence."
    if role == "core_support":
        return "direct_support: may be cited as support only for the exact fact stated in snippets/fact."
    if role == "core_refutation":
        return "direct_refutation: may be cited as refutation only for the exact fact stated in snippets/fact."

    if relation == "claim_origin":
        return "claim_origin_only: cite only to show that the claim was made or circulated; never use as proof."
    if stance == "context_only":
        return "context_only: cite only if essential; do not use as direct proof."
    return "use_cautiously: cite only exact packet-supported facts."


def normalize_candidate(item, tier, index):
    role = clean_text(item.get("evidence_role"))
    relation = clean_text(item.get("source_relation"))
    stance = clean_text(item.get("stance_to_claim"))
    flags = [clean_text(x) for x in listify(item.get("risk_flags")) if clean_text(x)]
    snippets = []
    seen_snips = set()

    for snip in item.get("snippets", []) or []:
        if not isinstance(snip, dict):
            continue
        text = clean_text(snip.get("text"), 600)
        if not text:
            continue
        key = text.lower()
        if key in seen_snips:
            continue
        seen_snips.add(key)
        snippets.append({
            "snippet_type": clean_text(snip.get("snippet_type")),
            "text": text,
            "stance": clean_text(snip.get("stance")),
            "snippet_score": safe_int(snip.get("snippet_score"), 0),
        })

    return {
        "stage3_tier": tier,
        "stage3_index": index,
        "url": clean_text(item.get("url")),
        "file_name": clean_text(item.get("file_name")),
        "source_relation": relation,
        "evidence_role": role,
        "claim_match_score": safe_int(item.get("claim_match_score"), 0),
        "stance_to_claim": stance,
        "safe_source_fact": clean_text(item.get("safe_source_fact"), 700),
        "snippets": snippets[:3],
        "risk_flags": flags,
        "caveats": [clean_text(x) for x in listify(item.get("caveats")) if clean_text(x)],
        "selection_reason": clean_text(item.get("selection_reason")),
        "allowed_use": source_usage_rule(item),
    }


def is_stage3_minimum_promotion(item):
    reason = clean_text(item.get("selection_reason")).lower()
    return "promoted from optional to meet compact minimum evidence packet" in reason


def exclusion_reason(item, rating):
    role = clean_text(item.get("evidence_role"))
    relation = clean_text(item.get("source_relation"))
    score = safe_int(item.get("claim_match_score"), 0)
    flags = {flag_prefix(f) for f in item.get("risk_flags", [])}

    if not item.get("url"):
        return "missing URL"
    if role in BAD_ROLES or relation in {"off_topic", "no_content"}:
        return "off-topic or no readable claim-useful content"
    if flags & HARD_EXCLUDE_FLAG_PREFIXES:
        return "hard risk flag: " + ", ".join(sorted(flags & HARD_EXCLUDE_FLAG_PREFIXES))
    if not item_has_usable_text(item):
        return "no usable safe_source_fact or snippet"
    if score <= 1:
        return "claim match score too low"
    if "weak_context_only" in flags:
        return "weak_context_only flag"
    if "boilerplate_or_sidebar" in flags and role not in CORE_ROLES | PROVENANCE_ROLES:
        return "boilerplate/sidebar risk without strong direct role"

    # Stage 3 sometimes promoted optional context only to meet a minimum.
    # Do not let that promotion survive unless the underlying evidence is independently strong.
    if is_stage3_minimum_promotion(item):
        if role not in CORE_ROLES | PROVENANCE_ROLES | CAVEAT_ROLES | STAT_ROLES | {"satirical_origin", "fake_news_origin"}:
            return "Stage 3 minimum-count promotion of non-core evidence"
        if score < 4:
            return "Stage 3 minimum-count promotion below strong-evidence threshold"

    # Background is normally unsafe as article proof. Keep only if nuance needs it and no better caveat exists.
    if role == "background_context" and rating not in NUANCE_RATINGS:
        return "background_context not essential for this rating"
    if role == "background_context" and score < 4:
        return "background_context below strong context threshold"

    return ""


def base_role_weight(role):
    if role in CORE_ROLES:
        return 100
    if role in {"visual_misattribution", "image_manipulation"}:
        return 98
    if role == "visual_origin":
        return 90
    if role in {"satirical_origin", "fake_news_origin"}:
        return 92
    if role in CAVEAT_ROLES:
        return 84
    if role in STAT_ROLES:
        return 82
    if role == "claim_origin":
        return 45
    if role in {"legal_or_definition_context", "scientific_context"}:
        return 40
    if role == "background_context":
        return 15
    return 0


def rating_bonus(item, rating):
    role = item.get("evidence_role")
    stance = item.get("stance_to_claim")
    bonus = 0

    if rating in FALSE_RATINGS:
        if role in {"core_refutation", "visual_misattribution", "image_manipulation", "fake_news_origin", "satirical_origin", "temporal_refutation"}:
            bonus += 20
        if stance in {"refutes_claim", "refutes_supporting_evidence", "satire"}:
            bonus += 10

    elif rating in {"true", "mostly_true"}:
        if role in {"core_support", "statistical_evidence"}:
            bonus += 18
        if rating == "mostly_true" and role in CAVEAT_ROLES | {"legal_or_definition_context", "scientific_context", "background_context"}:
            bonus += 16

    elif rating in {"half_true", "misleading"}:
        if role in {"core_support", "core_refutation", "rating_caveat", "temporal_refutation", "statistical_evidence"}:
            bonus += 18
        if stance in {"qualifies_claim", "refutes_claim", "refutes_supporting_evidence"}:
            bonus += 10

    elif rating == "satire":
        if role == "satirical_origin":
            bonus += 25
        if role in {"claim_origin", "fake_news_origin"}:
            bonus += 8

    return bonus


def candidate_rank_score(item, rating):
    role = item.get("evidence_role")
    stance = item.get("stance_to_claim")
    score = safe_int(item.get("claim_match_score"), 0)
    flags = {flag_prefix(f) for f in item.get("risk_flags", [])}

    val = base_role_weight(role) + (score * 8) + rating_bonus(item, rating)

    if stance in {"supports_claim", "refutes_claim", "refutes_supporting_evidence", "qualifies_claim", "satire"}:
        val += 8
    if item.get("safe_source_fact"):
        val += 4
    if item.get("snippets"):
        val += 3

    if role == "claim_origin":
        val -= 15
    if role in CONTEXT_ROLES:
        val -= 10
    if flags & SOFT_EXCLUDE_FLAG_PREFIXES:
        val -= 20
    if "claim_origin_not_proof" in flags:
        val -= 4
    if is_stage3_minimum_promotion(item):
        val -= 40

    return val


def dedupe_candidates(candidates):
    # 1. Keep highest-ranked item per exact URL.
    by_url = {}
    for item in candidates:
        url = item.get("url", "")
        if not url:
            continue
        prev = by_url.get(url)
        if prev is None or item.get("rank_score", 0) > prev.get("rank_score", 0):
            by_url[url] = item

    # 2. Remove near-duplicate facts.
    out = []
    seen_facts = set()
    for item in sorted(by_url.values(), key=lambda x: (-x.get("rank_score", 0), x.get("url", ""))):
        blob = clean_text(item.get("safe_source_fact") or item_text_blob(item)).lower()
        fact_key = re.sub(r"[^a-z0-9]+", " ", blob)[:180]
        if fact_key and fact_key in seen_facts:
            item["stage4_exclusion_reason"] = "duplicate safe_source_fact/snippet of stronger source"
            continue
        if fact_key:
            seen_facts.add(fact_key)
        out.append(item)
    return out


def evidence_bucket(item):
    role = item.get("evidence_role")
    stance = item.get("stance_to_claim")
    if role == "claim_origin":
        return "origin"
    if role in {"satirical_origin", "fake_news_origin"}:
        return "origin_refutation"
    if role in PROVENANCE_ROLES:
        return "provenance"
    if role in {"core_support"} or stance == "supports_claim":
        return "support"
    if role in {"core_refutation"} or stance in {"refutes_claim", "refutes_supporting_evidence", "satire"}:
        return "refutation"
    if role in CAVEAT_ROLES or stance == "qualifies_claim":
        return "caveat"
    if role in STAT_ROLES:
        return "statistical"
    if role in CONTEXT_ROLES:
        return "context"
    return "other"


def select_defensive_evidence(packet):
    rating = normalize_rating(packet.get("normalized_rating") or packet.get("original_rating"))
    raw_candidates = []

    for tier in ["must_cite_evidence", "optional_context"]:
        for idx, item in enumerate(packet.get(tier, []) or []):
            if isinstance(item, dict):
                raw_candidates.append(normalize_candidate(item, tier, idx))

    excluded = []
    kept = []

    for item in raw_candidates:
        reason = exclusion_reason(item, rating)
        item["stage4_bucket"] = evidence_bucket(item)

        if reason:
            item["stage4_exclusion_reason"] = reason
            excluded.append(item)
            continue

        item["rank_score"] = candidate_rank_score(item, rating)
        kept.append(item)

    ranked = dedupe_candidates(kept)

    selected = []
    selected_urls = set()

    def add_item(item, reason):
        url = item.get("url")
        if not url or url in selected_urls:
            return False
        if len(selected) >= MAX_STAGE4_EVIDENCE:
            return False
        item = dict(item)
        item["stage4_selection_reason"] = reason
        selected.append(item)
        selected_urls.add(url)
        return True

    # False/pants-fire/satire/visual claims should start with refutation/provenance/origin evidence.
    if rating in FALSE_RATINGS | {"satire"}:
        priority_buckets = {"refutation", "provenance", "origin_refutation"}
        for item in ranked:
            if item.get("stage4_bucket") in priority_buckets:
                add_item(item, "rating-priority refutation/provenance/origin evidence")
                if len(selected) >= 2:
                    break

    # Nuanced ratings need both the plausible/supported part and the caveat/context/refuting part.
    if rating in NUANCE_RATINGS:
        has_support = any(x.get("stage4_bucket") in {"support", "statistical"} for x in selected)
        has_caveat = any(x.get("stage4_bucket") in {"caveat", "refutation", "provenance", "context"} for x in selected)

        if not has_support:
            for item in ranked:
                if item.get("stage4_bucket") in {"support", "statistical"}:
                    add_item(item, "nuance requirement: supported/plausible part")
                    break

        if not has_caveat:
            for item in ranked:
                if item.get("stage4_bucket") in {"caveat", "refutation", "provenance", "context"}:
                    add_item(item, "nuance requirement: caveat/context/refuting part")
                    break

    # Statistical claims should include exact statistical evidence if available.
    for item in ranked:
        if item.get("stage4_bucket") == "statistical":
            add_item(item, "statistical evidence priority")
            break

    # Fill remaining slots by rank. This is a cap, not a minimum.
    for item in ranked:
        add_item(item, "top-ranked after defensive filtering")

    # Keep at most one ordinary claim-origin source unless it is satire/fake provenance.
    final = []
    origin_count = 0
    for item in selected:
        if item.get("evidence_role") == "claim_origin":
            origin_count += 1
            if origin_count > 1:
                item["stage4_exclusion_reason"] = "duplicate ordinary claim-origin source"
                excluded.append(item)
                continue
        final.append(item)

    # If still too many after origin pruning, recut by rank.
    final = sorted(final, key=lambda x: (-x.get("rank_score", 0), x.get("url", "")))[:MAX_STAGE4_EVIDENCE]

    warnings = []
    buckets = Counter(x.get("stage4_bucket") for x in final)
    roles = Counter(x.get("evidence_role") for x in final)

    if not final:
        warnings.append("No evidence survived Stage 4 defensive filtering; article must be very cautious.")
    if rating in NUANCE_RATINGS:
        if not any(b in buckets for b in ["support", "statistical"]):
            warnings.append("Nuanced rating lacks selected support/statistical evidence for the plausible part.")
        if not any(b in buckets for b in ["caveat", "refutation", "provenance", "context"]):
            warnings.append("Nuanced rating lacks selected caveat/refutation/context evidence.")
    if rating in FALSE_RATINGS and not any(b in buckets for b in ["refutation", "provenance", "origin_refutation"]):
        warnings.append("False rating lacks selected strong refutation/provenance/origin-refutation evidence.")
    if roles.get("claim_origin"):
        warnings.append("Claim-origin evidence is selected only to show circulation, not proof.")
    if any(x.get("evidence_role") in STAT_ROLES for x in final):
        warnings.append("Use exact statistical claims only when exact number/unit/framing appears in selected evidence.")

    return {
        "id": claim_id(packet),
        "claim": clean_text(packet.get("claim")),
        "original_rating": clean_text(packet.get("original_rating")),
        "normalized_rating": rating,
        "verdict_logic": [clean_text(x) for x in listify(packet.get("verdict_logic")) if clean_text(x)],
        "selected_evidence": final,
        "stage4_excluded_evidence": excluded,
        "stage4_filter_warnings": warnings,
        "candidate_count": len(raw_candidates),
        "selected_count": len(final),
        "excluded_count": len(excluded),
    }

## 6. Inspect defensive filtering for one claim

In [ ]:
packet_map = {claim_id(p): p for p in stage3_packets}

if INSPECT_ID not in packet_map:
    print("No packet found for", INSPECT_ID)
else:
    filtered = select_defensive_evidence(packet_map[INSPECT_ID])
    print("Claim:", filtered["claim"])
    print("Rating:", filtered["normalized_rating"])
    print("Candidates:", filtered["candidate_count"], "Selected:", filtered["selected_count"], "Excluded:", filtered["excluded_count"])
    print("Warnings:", filtered["stage4_filter_warnings"])
    print("=" * 100)

    print("\nSELECTED EVIDENCE")
    for i, item in enumerate(filtered["selected_evidence"], start=1):
        print(f"\n[{i}] {item.get('url')}")
        print(" role:", item.get("evidence_role"), "| stance:", item.get("stance_to_claim"), "| score:", item.get("claim_match_score"), "| rank:", item.get("rank_score"))
        print(" bucket:", item.get("stage4_bucket"))
        print(" allowed_use:", item.get("allowed_use"))
        print(" selection:", item.get("stage4_selection_reason"))
        print(" fact:", item.get("safe_source_fact"))
        if item.get("risk_flags"):
            print(" flags:", item.get("risk_flags"))

    print("\nEXCLUDED EVIDENCE")
    for i, item in enumerate(filtered["stage4_excluded_evidence"][:20], start=1):
        print(f"\n[{i}] {item.get('url')}")
        print(" role:", item.get("evidence_role"), "| score:", item.get("claim_match_score"), "| tier:", item.get("stage3_tier"))
        print(" excluded:", item.get("stage4_exclusion_reason"))
        print(" reason:", item.get("selection_reason"))
        print(" fact:", item.get("safe_source_fact"))

## 7. Sentence-plan prompt builder and API functions

This version prevents citation hallucination by **not letting the model write URLs or citation strings**.

The model returns a structured sentence plan:

```json
{
  "id": "30356",
  "verdict": "misleading",
  "sentences": [
    {"text": "Evidence-backed sentence with no URL typed by the model", "evidence_ids": ["E1"]}
  ]
}
```

Python then deterministically appends exact inline citations from the selected evidence map:

```text
Evidence-backed sentence with no URL typed by the model (source: https://example.com).
```

This protects against invented/modified URLs and makes citation behavior easier to inspect.


In [ ]:

STAGE4_PLAN_SCHEMA = {
    "type": "object",
    "properties": {
        "id": {"type": "string"},
        "verdict": {"type": "string"},
        "sentences": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "text": {"type": "string"},
                    "evidence_ids": {
                        "type": "array",
                        "items": {"type": "string"},
                    },
                    "purpose": {"type": "string"},
                },
                "required": ["text", "evidence_ids"],
            },
        },
    },
    "required": ["id", "verdict", "sentences"],
}

# Because the official citation precision script gives multi-URL citations precision=0
# if any URL is unnecessary, this notebook strongly prefers one evidence URL per sentence.
MAX_EVIDENCE_IDS_PER_SENTENCE = 1
MIN_CITED_SENTENCES = 1


def make_stage4_config(temperature, use_thinking=True):
    kwargs = dict(
        temperature=temperature,
        response_mime_type="application/json",
        response_schema=STAGE4_PLAN_SCHEMA,
        max_output_tokens=STAGE4_MAX_OUTPUT_TOKENS,
    )

    if use_thinking and STAGE4_THINKING_LEVEL:
        kwargs["thinking_config"] = types.ThinkingConfig(
            thinking_level=STAGE4_THINKING_LEVEL
        )

    return types.GenerateContentConfig(**kwargs)


def evidence_catalog(selected_evidence):
    """Assign deterministic short IDs to selected evidence for the model.

    The model sees E1/E2/etc. Python owns the URL mapping and appends citations later.
    """
    catalog = []
    for i, item in enumerate(selected_evidence, start=1):
        snippets = []
        for snip in item.get("snippets", []) or []:
            if not isinstance(snip, dict):
                continue
            snippets.append({
                "text": clean_text(snip.get("text"), 500),
                "stance": clean_text(snip.get("stance")),
                "snippet_type": clean_text(snip.get("snippet_type")),
                "snippet_score": safe_int(snip.get("snippet_score"), 0),
            })

        catalog.append({
            "evidence_id": f"E{i}",
            "url": item.get("url"),
            "evidence_role": item.get("evidence_role"),
            "source_relation": item.get("source_relation"),
            "stance_to_claim": item.get("stance_to_claim"),
            "claim_match_score": item.get("claim_match_score"),
            "allowed_use": item.get("allowed_use"),
            "safe_source_fact": item.get("safe_source_fact"),
            "snippets": snippets,
            "risk_flags": item.get("risk_flags", []),
            "stage4_bucket": item.get("stage4_bucket"),
        })
    return catalog


def build_stage4_prompt(filtered_packet):
    catalog = evidence_catalog(filtered_packet.get("selected_evidence", []))

    return f"""You are writing a concise fact-check article for CheckThat Task 3.

Important citation-control rule:
- Do NOT write URLs.
- Do NOT write citation text such as "(source: ...)".
- Return only a JSON sentence plan.
- Each sentence object must contain plain text plus evidence_ids such as ["E1"].
- Python will append exact URLs later. This prevents URL hallucination.

Use only the selected Stage 4 evidence below. Do not use outside knowledge. Do not use any reference/ground-truth article.

The Stage 3 packet may have been noisy, so follow the Stage 4 evidence filtering strictly:
- selected_evidence is the only evidence you may use.
- Do not cite or rely on excluded evidence.
- Claim-origin sources may only show what was claimed or circulated. They are never proof that the real-world claim is true.
- Context-only or background sources are not proof. Use them only when essential for caveat/context.
- If evidence is insufficient, write cautiously and say only what the selected evidence establishes.
- Do not invent missing evidence, dates, numbers, named entities, explanations, or causal claims.

Sentence/citation planning rules:
- Every evidence-based factual sentence must have evidence_ids.
- Prefer exactly one evidence_id per sentence.
- Split multi-source statements into separate sentences whenever possible.
- Use multiple evidence_ids only if every evidence source is necessary for that exact sentence.
- Do not include raw URLs, markdown links, footnotes, [1], bibliographies, or file names.
- Keep the article concise: roughly 120-240 words.
- Avoid long generic introductions.
- Avoid saying "the selected evidence" or "the packet" in the final article text.

Verdict/rating rules:
- Preserve the original rating/nuance: {filtered_packet.get("normalized_rating")}.
- For mostly_true, half_true, or misleading: explain both what part is supported/plausible and what caveat/context changes the interpretation.
- For false or pants_fire: lead with the strongest selected refutation, provenance evidence, or contradiction.
- For visual/image/video/screenshot claims: prioritize provenance, misattribution, manipulation, or origin evidence.
- For statistical claims: do not state exact numbers unless the selected evidence contains exact numbers, units, denominator, and framing.

Return strict JSON only with this shape:
{{
  "id": "{filtered_packet.get("id")}",
  "verdict": "{filtered_packet.get("normalized_rating")}",
  "sentences": [
    {{"text": "Plain sentence without citation or URL.", "evidence_ids": ["E1"], "purpose": "refutation/support/caveat/context/origin/conclusion"}}
  ]
}}

Claim ID: {filtered_packet.get("id")}
Claim: {filtered_packet.get("claim")}
Original rating: {filtered_packet.get("original_rating")}
Normalized rating: {filtered_packet.get("normalized_rating")}

Verdict logic from Stage 3, for guidance only:
{json.dumps(filtered_packet.get("verdict_logic", []), ensure_ascii=False, indent=2)}

Selected evidence catalog. Use evidence_id values only; do not copy URLs into your answer:
{json.dumps(catalog, ensure_ascii=False, indent=2)}

Stage 4 filter warnings:
{json.dumps(filtered_packet.get("stage4_filter_warnings", []), ensure_ascii=False, indent=2)}
"""


def call_stage4_model(prompt, temperature, repair=False):
    last_error = None
    use_thinking = True
    model_candidates = [STAGE4_MODEL] + [m for m in STAGE4_FALLBACK_MODELS if m != STAGE4_MODEL]

    for model_name in model_candidates:
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                response = client.models.generate_content(
                    model=model_name,
                    contents=prompt,
                    config=make_stage4_config(temperature=temperature, use_thinking=use_thinking),
                )
                parsed = extract_json_response(response)
                parsed["_model_used"] = model_name + ("/repair" if repair else "")
                parsed["_usage"] = response_usage_summary(response)
                return parsed

            except Exception as e:
                last_error = repr(e)

                if use_thinking and "thinking" in last_error.lower():
                    print("thinking_config rejected; retrying without thinking_config")
                    use_thinking = False
                    continue

                sleep_s = min(2 ** attempt, 20) + random.random()
                print(f"Stage 4 model={model_name} attempt {attempt}/{MAX_RETRIES} failed: {last_error}. Sleeping {sleep_s:.1f}s")
                time.sleep(sleep_s)

        print(f"Trying fallback model after failures with {model_name}")

    raise RuntimeError(f"All Stage 4 models failed. Last error: {last_error}")


def build_repair_prompt(filtered_packet, bad_plan, warnings):
    catalog = evidence_catalog(filtered_packet.get("selected_evidence", []))
    return f"""Repair this CheckThat Stage 4 sentence plan.

Do not add new facts. Do not use outside knowledge.
Do not write URLs or citation strings. Python will append exact source URLs later.
Use only evidence_ids from the catalog.
Prefer exactly one evidence_id per sentence. Split multi-source facts into separate sentences.

Validation warnings/errors:
{json.dumps(warnings, ensure_ascii=False, indent=2)}

Claim: {filtered_packet.get("claim")}
Expected ID: {filtered_packet.get("id")}
Expected verdict: {filtered_packet.get("normalized_rating")}

Evidence catalog:
{json.dumps(catalog, ensure_ascii=False, indent=2)}

Bad sentence plan:
{json.dumps(bad_plan, ensure_ascii=False, indent=2)}

Return corrected strict JSON only:
{{
  "id": "{filtered_packet.get("id")}",
  "verdict": "{filtered_packet.get("normalized_rating")}",
  "sentences": [
    {{"text": "Plain sentence without citation or URL.", "evidence_ids": ["E1"], "purpose": "refutation/support/caveat/context/origin/conclusion"}}
  ]
}}
"""


## 8. Deterministic citation composition, validation, and fallback

The model returns evidence IDs, not citations. This cell:

1. validates the sentence plan,
2. strips any accidental URLs/citations from model text,
3. appends exact inline citations from Python’s evidence map,
4. validates final citation format and exact URL membership,
5. provides a deterministic fallback if the model still fails.


In [ ]:

CITATION_RE = re.compile(r"\(source:\s*([^)]+?)\)", flags=re.IGNORECASE)
URL_RE = re.compile(r"https?://[^\s\)\]\}\>,;\"']+")
TERMINAL_PUNCT_RE = re.compile(r"[.!?]+$")


def extract_cited_urls(article):
    cited = []
    for match in CITATION_RE.finditer(article or ""):
        blob = match.group(1)
        for part in blob.split(";"):
            url = part.strip()
            if url:
                cited.append(url)
    return unique_keep_order(cited)


def extract_all_urls(article):
    return unique_keep_order(URL_RE.findall(article or ""))


def id_to_url_map(filtered_packet):
    return {row["evidence_id"]: row["url"] for row in evidence_catalog(filtered_packet.get("selected_evidence", [])) if row.get("url")}


def url_to_selected_map(filtered_packet):
    return {x.get("url"): x for x in filtered_packet.get("selected_evidence", []) if x.get("url")}


def sanitize_sentence_text(text):
    text = clean_text(text)
    # Remove accidental citations/URLs because Python owns citation insertion.
    text = CITATION_RE.sub("", text)
    text = URL_RE.sub("", text)
    text = re.sub(r"\[[0-9]+\]", "", text)
    text = re.sub(r"\[[^\]]+\]\(\s*\)", "", text)
    text = clean_text(text)
    # Remove trailing punctuation before appending deterministic citation.
    text = TERMINAL_PUNCT_RE.sub("", text).strip()
    return text


def validate_stage4_plan(plan, filtered_packet):
    warnings = []
    cid = filtered_packet.get("id")
    rating = filtered_packet.get("normalized_rating")
    id_to_url = id_to_url_map(filtered_packet)

    if not isinstance(plan, dict):
        return ["ERROR: plan is not a JSON object"]

    if clean_text(plan.get("id")) != cid:
        warnings.append(f"ERROR: id mismatch; expected {cid}, got {plan.get('id')}")
    if clean_text(plan.get("verdict")) != rating:
        warnings.append(f"ERROR: verdict mismatch; expected {rating}, got {plan.get('verdict')}")

    sentences = plan.get("sentences")
    if not isinstance(sentences, list) or not sentences:
        warnings.append("ERROR: plan has no sentences")
        return warnings

    cited_sentence_count = 0
    for idx, sent in enumerate(sentences, start=1):
        if not isinstance(sent, dict):
            warnings.append(f"ERROR: sentence {idx} is not an object")
            continue

        text = clean_text(sent.get("text"))
        if not text:
            warnings.append(f"ERROR: sentence {idx} has empty text")
            continue

        if URL_RE.search(text) or CITATION_RE.search(text):
            warnings.append(f"ERROR: sentence {idx} contains URL/citation text; model must only output plain text")
        if re.search(r"\[[0-9]+\]", text):
            warnings.append(f"ERROR: sentence {idx} contains bracket citation format")
        if re.search(r"\[[^\]]+\]\(https?://", text):
            warnings.append(f"ERROR: sentence {idx} contains markdown link")

        evidence_ids = sent.get("evidence_ids", [])
        if not isinstance(evidence_ids, list):
            warnings.append(f"ERROR: sentence {idx} evidence_ids is not a list")
            continue

        evidence_ids = unique_keep_order([clean_text(e) for e in evidence_ids if clean_text(e)])
        if evidence_ids:
            cited_sentence_count += 1

        bad_ids = [e for e in evidence_ids if e not in id_to_url]
        if bad_ids:
            warnings.append(f"ERROR: sentence {idx} uses invalid evidence_ids: {bad_ids}")

        if len(evidence_ids) > MAX_EVIDENCE_IDS_PER_SENTENCE:
            warnings.append(
                f"ERROR: sentence {idx} cites {len(evidence_ids)} evidence IDs; split into one-source sentences for citation precision"
            )

    if id_to_url and cited_sentence_count < MIN_CITED_SENTENCES:
        warnings.append("ERROR: no evidence-backed sentences in plan")

    return warnings


def compose_article_from_plan(plan, filtered_packet):
    id_to_url = id_to_url_map(filtered_packet)
    sentences_out = []
    used_urls = []
    sentence_plan_out = []

    for sent in plan.get("sentences", []) or []:
        if not isinstance(sent, dict):
            continue

        text = sanitize_sentence_text(sent.get("text", ""))
        if not text:
            continue

        raw_ids = sent.get("evidence_ids", [])
        if not isinstance(raw_ids, list):
            raw_ids = []
        evidence_ids = unique_keep_order([clean_text(e) for e in raw_ids if clean_text(e) in id_to_url])

        # If validation is bypassed, still enforce the single-source preference here.
        # The repair pass should normally split these before composition.
        if len(evidence_ids) > MAX_EVIDENCE_IDS_PER_SENTENCE:
            evidence_ids = evidence_ids[:MAX_EVIDENCE_IDS_PER_SENTENCE]

        urls = unique_keep_order([id_to_url[e] for e in evidence_ids if id_to_url.get(e)])
        if urls:
            citation = "; ".join(urls)
            sentence_text = f"{text} (source: {citation})."
            used_urls.extend(urls)
        else:
            sentence_text = f"{text}."

        sentences_out.append(sentence_text)
        sentence_plan_out.append({
            "text": text,
            "evidence_ids": evidence_ids,
            "urls": urls,
            "purpose": clean_text(sent.get("purpose")),
        })

    return clean_text(" ".join(sentences_out)), unique_keep_order(used_urls), sentence_plan_out


def selected_expected_urls(filtered_packet):
    """Strong URLs we especially want cited, without forcing weak/context-only URLs."""
    expected = []
    for item in filtered_packet.get("selected_evidence", []):
        role = item.get("evidence_role")
        score = safe_int(item.get("claim_match_score"), 0)
        bucket = item.get("stage4_bucket")

        if role in CORE_ROLES | PROVENANCE_ROLES | {"satirical_origin", "fake_news_origin"} | CAVEAT_ROLES | STAT_ROLES:
            if score >= 4 or bucket in {"refutation", "provenance", "origin_refutation", "support", "caveat", "statistical"}:
                expected.append(item.get("url"))

    return unique_keep_order([u for u in expected if u])


def validate_stage4_article(article, used_sources, filtered_packet, sentence_plan=None):
    warnings = []
    article = clean_text(article)

    if not article:
        warnings.append("ERROR: article is empty")

    allowed_urls = {x.get("url") for x in filtered_packet.get("selected_evidence", []) if x.get("url")}
    allowed_file_names = {x.get("file_name") for x in filtered_packet.get("selected_evidence", []) if x.get("file_name")}

    cited_urls = extract_cited_urls(article)
    all_urls = extract_all_urls(article)

    if article and not cited_urls and allowed_urls:
        warnings.append("ERROR: article has no valid inline source citations")

    bad_cited = [u for u in cited_urls if u not in allowed_urls]
    if bad_cited:
        warnings.append("ERROR: cited URL not in selected allowed URLs: " + "; ".join(bad_cited))

    uncited_raw_urls = [u for u in all_urls if u not in cited_urls]
    if uncited_raw_urls:
        warnings.append("ERROR: raw URL appears outside valid citation format: " + "; ".join(uncited_raw_urls[:5]))

    invented_urls = [u for u in all_urls if u not in allowed_urls]
    if invented_urls:
        warnings.append("ERROR: article contains invented/modified URL: " + "; ".join(invented_urls[:5]))

    for fname in allowed_file_names:
        if fname and fname in article:
            warnings.append(f"ERROR: article cites or mentions file name: {fname}")

    if re.search(r"\[[0-9]+\]", article):
        warnings.append("ERROR: article uses bracket citation format like [1]")
    if re.search(r"\[[^\]]+\]\(https?://", article):
        warnings.append("ERROR: article uses markdown links")

    # The evaluator's multi-citation precision is strict. Warn/error if a sentence has multiple URLs.
    for c in CITATION_RE.finditer(article):
        urls = [u.strip() for u in c.group(1).split(";") if u.strip()]
        if len(urls) > MAX_EVIDENCE_IDS_PER_SENTENCE:
            warnings.append("ERROR: multi-URL citation found; split the sentence to protect citation precision")

    expected = selected_expected_urls(filtered_packet)
    missing_expected = [u for u in expected if u not in cited_urls]
    if missing_expected and len(cited_urls) < max(MIN_STRONG_SELECTED_FOR_REPAIR, min(len(expected), 3)):
        warnings.append("ERROR: too few selected strong evidence URLs were cited: " + "; ".join(missing_expected[:5]))
    elif missing_expected:
        warnings.append("WARNING: selected strong evidence not cited: " + "; ".join(missing_expected[:5]))

    selected_by_url = url_to_selected_map(filtered_packet)
    cited_roles = Counter(selected_by_url.get(u, {}).get("evidence_role") for u in cited_urls)
    rating = filtered_packet.get("normalized_rating")

    if cited_urls and set(cited_roles.keys()) <= {"claim_origin"}:
        warnings.append("ERROR: article cites only ordinary claim-origin evidence; this is not proof")

    if rating in NUANCE_RATINGS:
        has_core = any(selected_by_url.get(u, {}).get("evidence_role") in CORE_ROLES for u in cited_urls)
        has_caveat = any(selected_by_url.get(u, {}).get("evidence_role") in CAVEAT_ROLES | CONTEXT_ROLES for u in cited_urls)
        if not (has_core and has_caveat):
            warnings.append("WARNING: nuanced rating should cite both support/plausibility and caveat/context when available")

    if rating in FALSE_RATINGS:
        has_refute = any(
            selected_by_url.get(u, {}).get("evidence_role") in {"core_refutation", "visual_misattribution", "image_manipulation", "fake_news_origin", "satirical_origin", "temporal_refutation"}
            for u in cited_urls
        )
        if not has_refute:
            warnings.append("WARNING: false/pants_fire rating has no cited refutation/provenance/fake-origin evidence")

    if set(cited_urls) != set(used_sources):
        warnings.append("ERROR: used_sources does not match URLs parsed from article citations")

    return warnings


def needs_repair(warnings):
    return any(str(w).startswith("ERROR:") for w in warnings)


def make_record_from_plan(plan, filtered_packet, repaired=False, extra_warnings=None):
    article, used_sources, sentence_plan = compose_article_from_plan(plan, filtered_packet)
    warnings = []
    warnings.extend(validate_stage4_plan(plan, filtered_packet))
    warnings.extend(validate_stage4_article(article, used_sources, filtered_packet, sentence_plan=sentence_plan))
    if extra_warnings:
        warnings.extend(extra_warnings)

    return {
        "id": filtered_packet.get("id"),
        "claim": filtered_packet.get("claim"),
        "original_rating": filtered_packet.get("original_rating"),
        "verdict": filtered_packet.get("normalized_rating"),
        "article": article,
        "used_sources": used_sources,
        "sentence_plan": sentence_plan,
        "selected_evidence": filtered_packet.get("selected_evidence", []),
        "stage4_excluded_evidence": filtered_packet.get("stage4_excluded_evidence", []),
        "stage4_filter_warnings": filtered_packet.get("stage4_filter_warnings", []),
        "validation_warnings": warnings,
        "repaired": repaired,
        "model": plan.get("_model_used"),
        "usage": plan.get("_usage", {}),
        "stage": "stage4_defensive_article_generation_manual_citations",
        "run_timestamp_utc": utc_now(),
    }


def deterministic_fallback_plan(filtered_packet):
    cid = filtered_packet.get("id")
    rating = filtered_packet.get("normalized_rating")
    claim = clean_text(filtered_packet.get("claim"))
    selected = filtered_packet.get("selected_evidence", [])
    catalog = evidence_catalog(selected)

    sentences = []
    if claim:
        sentences.append({
            "text": f"The claim being checked is that {claim}",
            "evidence_ids": [],
            "purpose": "claim_statement",
        })

    # Create short, one-source sentences so precision is safer.
    for row in catalog[:MAX_STAGE4_EVIDENCE]:
        eid = row.get("evidence_id")
        role = row.get("evidence_role")
        fact = clean_text(row.get("safe_source_fact"), 350)
        if not fact:
            snippets = row.get("snippets") or []
            fact = clean_text(snippets[0].get("text") if snippets else "", 350)
        if not fact:
            continue

        if role == "core_refutation":
            text = f"A source used for checking the claim gives contrary evidence: {fact}"
            purpose = "refutation"
        elif role == "core_support":
            text = f"A source used for checking the claim supports part of the statement: {fact}"
            purpose = "support"
        elif role == "claim_origin":
            text = f"One source shows the claim or a version of it circulated: {fact}"
            purpose = "origin"
        elif role in {"satirical_origin", "fake_news_origin"}:
            text = f"The provenance evidence points to a satirical, fake-news, or fabricated origin: {fact}"
            purpose = "origin_refutation"
        elif role in PROVENANCE_ROLES:
            text = f"The visual or provenance evidence is central to checking the claim: {fact}"
            purpose = "provenance"
        elif role in CAVEAT_ROLES or row.get("stance_to_claim") == "qualifies_claim":
            text = f"A caveat source changes how the claim should be interpreted: {fact}"
            purpose = "caveat"
        elif role in CONTEXT_ROLES:
            text = f"A context source provides relevant background: {fact}"
            purpose = "context"
        elif role in STAT_ROLES:
            text = f"A statistical source provides the relevant numerical framing: {fact}"
            purpose = "statistical"
        else:
            text = f"One selected source says: {fact}"
            purpose = "evidence"

        sentences.append({"text": text, "evidence_ids": [eid], "purpose": purpose})

    if rating:
        sentences.append({
            "text": f"On the selected evidence, the safest verdict to preserve is {rating}",
            "evidence_ids": [],
            "purpose": "conclusion",
        })

    return {
        "id": cid,
        "verdict": rating,
        "sentences": sentences,
        "_model_used": "deterministic_fallback",
        "_usage": {},
    }


## 9. Generate one article smoke test

This smoke test generates a sentence plan, composes citations manually, validates the final article, and falls back deterministically if needed.


In [ ]:

def generate_one_article(packet):
    filtered_packet = select_defensive_evidence(packet)

    try:
        prompt = build_stage4_prompt(filtered_packet)
        plan = call_stage4_model(
            prompt,
            temperature=STAGE4_TEMPERATURE,
            repair=False,
        )

        initial_record = make_record_from_plan(plan, filtered_packet, repaired=False)
        repaired = False

        if needs_repair(initial_record.get("validation_warnings", [])):
            repair_prompt = build_repair_prompt(filtered_packet, plan, initial_record.get("validation_warnings", []))
            repaired_plan = call_stage4_model(
                repair_prompt,
                temperature=STAGE4_REPAIR_TEMPERATURE,
                repair=True,
            )
            repaired_record = make_record_from_plan(repaired_plan, filtered_packet, repaired=True)

            old_errors = [w for w in initial_record.get("validation_warnings", []) if str(w).startswith("ERROR:")]
            new_errors = [w for w in repaired_record.get("validation_warnings", []) if str(w).startswith("ERROR:")]

            if len(new_errors) <= len(old_errors):
                initial_record = repaired_record
                repaired = True

        if needs_repair(initial_record.get("validation_warnings", [])):
            fallback_plan = deterministic_fallback_plan(filtered_packet)
            fallback_record = make_record_from_plan(
                fallback_plan,
                filtered_packet,
                repaired=repaired,
                extra_warnings=["WARNING: deterministic fallback used after model validation errors"],
            )
            initial_record = fallback_record

        return initial_record

    except Exception as e:
        fallback_plan = deterministic_fallback_plan(filtered_packet)
        return make_record_from_plan(
            fallback_plan,
            filtered_packet,
            repaired=False,
            extra_warnings=[f"WARNING: deterministic fallback used after API failure: {repr(e)}"],
        )


if INSPECT_ID in packet_map:
    print("Generating smoke-test article for", INSPECT_ID)
    smoke = generate_one_article(packet_map[INSPECT_ID])
    print(json.dumps({
        "id": smoke["id"],
        "verdict": smoke["verdict"],
        "article": smoke["article"],
        "used_sources": smoke["used_sources"],
        "sentence_plan": smoke.get("sentence_plan", []),
        "validation_warnings": smoke["validation_warnings"],
        "filter_warnings": smoke["stage4_filter_warnings"],
        "model": smoke["model"],
        "repaired": smoke["repaired"],
    }, ensure_ascii=False, indent=2)[:10000])
else:
    print("No packet found for smoke-test ID:", INSPECT_ID)


## 10. Resume helpers

In [ ]:
def load_existing_stage4_outputs():
    if STAGE4_OUT.exists():
        print("Loading existing local Stage 4 output:", STAGE4_OUT)
        return load_json(STAGE4_OUT)
    if DRIVE_STAGE4_OUT.exists():
        print("Loading existing Drive Stage 4 output:", DRIVE_STAGE4_OUT)
        existing = load_json(DRIVE_STAGE4_OUT)
        save_json(STAGE4_OUT, existing)
        return existing
    return []


def sorted_outputs_from_map(output_by_id):
    return sorted(output_by_id.values(), key=lambda x: str(x.get("id", "")))


def save_stage4_progress(output_by_id):
    rows = sorted_outputs_from_map(output_by_id)
    save_json(STAGE4_OUT, rows)
    copy_to_drive(STAGE4_OUT, DRIVE_STAGE4_OUT)
    return rows


def export_prediction(rows):
    pred = []
    for row in sorted(rows, key=lambda x: str(x.get("id", ""))):
        rid = row.get("id")
        try:
            rid_out = int(rid)
        except Exception:
            rid_out = rid

        pred.append({
            "id": rid_out,
            "factchecking_article": clean_text(row.get("article")),
        })

    save_json(PREDICTION_TAGGED_OUT, pred)
    save_json(PREDICTION_CANONICAL_OUT, pred)
    return pred

## 11. Run Stage 4 generation with resume + parallelism

In [ ]:
existing = load_existing_stage4_outputs()
output_by_id = {str(x.get("id")): x for x in existing if str(x.get("id"))}

todo = []
for packet in stage3_packets:
    cid = claim_id(packet)
    if not cid:
        continue
    if cid in FORCE_REGENERATE_IDS or cid not in output_by_id:
        todo.append(packet)

if MAX_API_CALLS is not None:
    todo = todo[:MAX_API_CALLS]

print("Existing Stage 4 outputs:", len(existing))
print("Remaining Stage 4 API calls in this run:", len(todo))
print("MAX_WORKERS:", MAX_WORKERS)

completed_this_run = 0

if not todo:
    print("Nothing to do.")
elif MAX_WORKERS <= 1:
    for idx, packet in enumerate(todo, start=1):
        cid = claim_id(packet)
        print(f"\nGenerating {idx}/{len(todo)} | claim {cid}")
        result = generate_one_article(packet)
        output_by_id[cid] = result
        completed_this_run += 1

        if completed_this_run % SAVE_AFTER_EVERY_COMPLETED == 0:
            rows = save_stage4_progress(output_by_id)
            print(
                f"Saved {len(rows)}/{len(stage3_packets)} | "
                f"{cid} | citations={len(result.get('used_sources', []))} | "
                f"warnings={len(result.get('validation_warnings', []))} | "
                f"model={result.get('model')} | repaired={result.get('repaired')}"
            )
else:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_packet = {executor.submit(generate_one_article, packet): packet for packet in todo}

        for future in as_completed(future_to_packet):
            packet = future_to_packet[future]
            cid = claim_id(packet)

            try:
                result = future.result()
            except Exception as e:
                # Defensive final fallback. This path should rarely run because generate_one_article already catches API failures.
                filtered_packet = select_defensive_evidence(packet)
                fallback_obj = deterministic_fallback_article(filtered_packet)
                warnings = validate_stage4_output(fallback_obj, filtered_packet)
                warnings.append(f"WARNING: outer fallback used after unexpected failure: {repr(e)}")
                result = finalize_stage4_record(fallback_obj, filtered_packet, warnings, repaired=False)

            output_by_id[cid] = result
            completed_this_run += 1

            if completed_this_run % SAVE_AFTER_EVERY_COMPLETED == 0:
                rows = save_stage4_progress(output_by_id)
                print(
                    f"Saved {len(rows)}/{len(stage3_packets)} | "
                    f"completed_this_run={completed_this_run}/{len(todo)} | "
                    f"{cid} | citations={len(result.get('used_sources', []))} | "
                    f"warnings={len(result.get('validation_warnings', []))} | "
                    f"model={result.get('model')} | repaired={result.get('repaired')}"
                )

rows = save_stage4_progress(output_by_id)
pred = export_prediction(rows)

print("\nFinal Stage 4 rows:", len(rows))
print("Prediction rows:", len(pred))
print("Stage 4 local output:", STAGE4_OUT)
print("Stage 4 Drive output:", DRIVE_STAGE4_OUT)
print("Tagged prediction output:", PREDICTION_TAGGED_OUT)
print("Canonical prediction output:", PREDICTION_CANONICAL_OUT)

## 12. Validation summary

In [ ]:
stage4_rows = load_json(STAGE4_OUT)

total = len(stage4_rows)
with_errors = []
with_warnings = []
citation_counts = Counter()
models = Counter()
repairs = Counter()

for row in stage4_rows:
    warnings = row.get("validation_warnings", []) or []
    errors = [w for w in warnings if str(w).startswith("ERROR:")]
    if errors:
        with_errors.append((row.get("id"), errors))
    elif warnings:
        with_warnings.append((row.get("id"), warnings))

    citation_counts[len(row.get("used_sources", []) or [])] += 1
    models[row.get("model")] += 1
    repairs[str(row.get("repaired"))] += 1

print("Total articles:", total)
print("Rows with validation errors:", len(with_errors))
print("Rows with non-error warnings:", len(with_warnings))
print("Citation count distribution:", dict(citation_counts))
print("Models:", dict(models))
print("Repaired:", dict(repairs))

if with_errors:
    print("\nVALIDATION ERRORS")
    for cid, errors in with_errors[:30]:
        print("-", cid)
        for e in errors:
            print("   ", e)

if with_warnings:
    print("\nNON-ERROR WARNINGS")
    for cid, warnings in with_warnings[:30]:
        print("-", cid)
        for w in warnings[:5]:
            print("   ", w)

## 13. Inspect one generated article

In [ ]:
stage4_rows = load_json(STAGE4_OUT)
row_map = {str(x.get("id")): x for x in stage4_rows}

if INSPECT_ID not in row_map:
    print("No Stage 4 output found for", INSPECT_ID)
else:
    row = row_map[INSPECT_ID]
    print("ID:", row.get("id"))
    print("Verdict:", row.get("verdict"))
    print("Model:", row.get("model"), "| repaired:", row.get("repaired"))
    print("Used sources:", row.get("used_sources"))
    print("Validation warnings:", row.get("validation_warnings"))
    print("Filter warnings:", row.get("stage4_filter_warnings"))
    print("=" * 100)
    print(row.get("article"))

    print("\nSELECTED EVIDENCE USED BY STAGE 4")
    for i, ev in enumerate(row.get("selected_evidence", []), start=1):
        print(f"\n[{i}] {ev.get('url')}")
        print(" role:", ev.get("evidence_role"), "| bucket:", ev.get("stage4_bucket"), "| score:", ev.get("claim_match_score"))
        print(" allowed_use:", ev.get("allowed_use"))
        print(" fact:", ev.get("safe_source_fact"))
        if ev.get("risk_flags"):
            print(" flags:", ev.get("risk_flags"))

    print("\nEXCLUDED BY STAGE 4")
    for i, ev in enumerate(row.get("stage4_excluded_evidence", [])[:15], start=1):
        print(f"\n[{i}] {ev.get('url')}")
        print(" role:", ev.get("evidence_role"), "| score:", ev.get("claim_match_score"), "| tier:", ev.get("stage3_tier"))
        print(" exclusion:", ev.get("stage4_exclusion_reason"))
        print(" fact:", ev.get("safe_source_fact"))

## 14. Manual repair helper for selected IDs

In [ ]:
# Optional: use this if a few IDs still need reruns after inspection.
# Example:
# FORCE_REGENERATE_IDS = {"30356", "30358"}
# Then rerun cells 10 and 11.

def regenerate_ids(ids):
    ids = {str(x) for x in ids}
    packets = [p for p in stage3_packets if claim_id(p) in ids]
    existing = load_existing_stage4_outputs()
    output_by_id = {str(x.get("id")): x for x in existing if str(x.get("id"))}

    for packet in packets:
        cid = claim_id(packet)
        print("Regenerating", cid)
        output_by_id[cid] = generate_one_article(packet)
        rows = save_stage4_progress(output_by_id)
        export_prediction(rows)
        print("Saved", cid, "warnings:", output_by_id[cid].get("validation_warnings"))

    return sorted_outputs_from_map(output_by_id)

print("Helper loaded. Call regenerate_ids({'30356'}) if needed.")

## 15. Final export check

In [ ]:
rows = load_json(STAGE4_OUT)
pred = export_prediction(rows)

print("Exported prediction rows:", len(pred))
print("Tagged:", PREDICTION_TAGGED_OUT)
print("Canonical:", PREDICTION_CANONICAL_OUT)

print("\nFirst prediction row:")
print(json.dumps(pred[0] if pred else {}, ensure_ascii=False, indent=2)[:2000])